In [1]:
from elasticsearch import Elasticsearch
import pandas as pd

In [ ]:
es=Elasticsearch("https://192.168.yyy.XX:9200",basic_auth=("your_user","password"),
                ca_certs="path_to_cert/elasticsearch-ca.pem",
                max_retries=10, retry_on_timeout=True)

es.ping()

True

In [3]:
df=pd.read_csv("myntra_products_catalog.csv")
df.head()

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White


In [4]:
df.fillna("None",inplace=True)

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White
...,...,...,...,...,...,...,...,...
12486,10262843,Pepe Jeans Men Black Hammock Slim Fit Low-Rise...,Pepe Jeans,Men,1299,7,"Black dark wash 5-pocket low-rise jeans, clean...",Black
12487,10261721,Mochi Women Gold-Toned Solid Heels,Mochi,Women,1990,5,"A pair of gold-toned open toe heels, has regul...",Gold
12488,10261607,612 league Girls Navy Blue & White Printed Reg...,612 league,Girls,602,4,Navy Blue and White printed mid-rise denim sho...,Blue
12489,10266621,Bvlgari Men Aqva Pour Homme Marine Eau de Toil...,Bvlgari,Men,8950,2,Bvlgari Men Aqva Pour Homme Marine Eau de Toil...,None


In [ ]:
df.isna().value_counts()

In [6]:
from sentence_transformers import SentenceTransformer
model=SentenceTransformer('all-mpnet-base-v2')

/home/dl-user/mario/sampleEnv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [7]:
df["DescriptionVector"] = list(model.encode(df["Description"].tolist(), show_progress_bar=True))

Batches: 100%|██████████| 391/391 [01:57<00:00,  3.34it/s]


In [ ]:
from indexMapping import indexMapping
es.indices.create(index="all_products", mappings=indexMapping)

In [8]:
record_list=df.to_dict("records")

In [9]:
for record in record_list:
    try:
        es.index(index="all_products",document=record,id=record["ProductID"])
    except Exception as e:
        print(e)

In [11]:
input_keyword="Blue Shoes"
vector_of_input=model.encode(input_keyword)

In [13]:
query={
    "field":"DescriptionVector",
    "query_vector":vector_of_input,
    "k":2,
    "num_candidates":500
}

In [14]:
res=es.knn_search(index="all_products",knn=query,source=["ProductName","Description"])
res["hits"]["hits"]

/tmp/ipykernel_171723/1926129187.py:1: GeneralAvailabilityWarning: This API is in technical preview and may be changed or removed in a future release. Elastic will work to fix any issues, but features in technical preview are not subject to the support SLA of official GA features.
  res=es.knn_search(index="all_products",knn=query,source=["ProductName","Description"])
/tmp/ipykernel_171723/1926129187.py:1: ElasticsearchWarning: The kNN search API has been replaced by the `knn` option in the search API.
  res=es.knn_search(index="all_products",knn=query,source=["ProductName","Description"])


[{'_index': 'all_products',
  '_id': '10250653',
  '_score': 0.6333018,
  '_source': {'ProductName': 'FURO by Red Chief Men Blue Mesh Walking Shoes',
   'Description': 'A pair of blue walking sports shoes, has regular styling, lace-up detailMesh upperCushioned footbedTextured and patterned outsoleWarranty: 2 monthsWarranty provided by brand/manufacturer'}},
 {'_index': 'all_products',
  '_id': '10250647',
  '_score': 0.6333018,
  '_source': {'ProductName': 'FURO by Red Chief Men Blue Mesh Walking Shoes',
   'Description': 'A pair of blue walking sports shoes, has regular styling, lace-up detailMesh upperCushioned footbedTextured and patterned outsoleWarranty: 2 monthsWarranty provided by brand/manufacturer'}}]

| Feature  | Search API (BM25) | kNN Search            |
|----------|-------------------|----------------------|
| Type     | Keyword search    | Semantic search      |
| Data     | Text tokens       | Vectors              |
| Speed    | Very fast         | Slower               |
| Accuracy | Exact match       | Meaning-based        |
| Infra    | Simple            | Needs embedding model |

## Nested search example

### Document structure
```json
{
  "created_at": 1776246520,
  "updated_at": 1776246520,
  "images": [
    {
      "image_name": "Hello_enroll.jpg",
      "image_hash_name": "adcac.jpg"
    }
  ]
}


```json
{
  "query": {
    "nested": {
      "path": "images",
      "query": {
        "match": {
          "images.image_hash_name": "0ca7bd077a062ce9.jpg"
        }
      }
    }
  }
}

| Limitation Semantic        | Why it matters        |
|--------------------|----------------------|
| Token limit        | loses long context   |
| Semantic fuzziness | bad for exact matches|
| Compression        | info loss            |
| No interaction     | weaker ranking       |
| Expensive          | scaling issues       |